# Clean and prepare raw data for warehouse

In [1]:
!pip install "trino[sqlalchemy]"

## Looker Dataset:

## 1. distribution_centers

In [4]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.distribution_centers"
dist_center = pd.read_sql(query, con=engine)

print('Loaded!')

Loaded!


In [2]:
dist_center.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         10 non-null     int64  
 1   name       10 non-null     object 
 2   latitude   10 non-null     float64
 3   longitude  10 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 452.0+ bytes


In [4]:
dist_center.isnull().sum()

id           0
name         0
latitude     0
longitude    0
dtype: int64

In [5]:
dist_center.head()

,id,name,latitude,longitude
0,1,Memphis TN,35.1174,-89.9711
1,2,Chicago IL,41.8369,-87.6847
2,3,Houston TX,29.7604,-95.3698
3,4,Los Angeles CA,34.0500,-118.2500
4,5,New Orleans LA,29.9500,-90.0667


### split name to city, state

In [ ]:
dist_center[['city', 'state']] = dist_center['name'].str.rsplit(' ', n=1, expand=True)
dist_center = dist_center.drop(columns=['name'])


In [ ]:
dist_center

### replace NY/NJ --> NY

In [ ]:
dist_center['state'] = dist_center['state'].replace('NY/NJ', 'NY')

In [ ]:
dist_center['city'] = dist_center['city'].replace('Port Authority of New York/New Jersey', 'New York')

In [ ]:
dist_center

### load to warehouse

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import time

print(f"Write {len(dist_center):,} rows...")
start_time = time.time()

dist_center.to_sql(
    name='distribution_centers', 
    con=engine, 
    schema='warehouse',     
    index=False, 
    chunksize=3500,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f"Done! In: {round(end_time - start_time, 2)} seconds.")

## 2. events

In [46]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.events"
events = pd.read_sql(query, con=engine)

print('Loaded!')


Loaded!


In [47]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2431963 entries, 0 to 2431962
Data columns (total 13 columns):
 #   Column           Dtype              
---  ------           -----              
 0   id               int64              
 1   user_id          float64            
 2   sequence_number  int64              
 3   session_id       object             
 4   created_at       datetime64[ns, UTC]
 5   ip_address       object             
 6   city             object             
 7   state            object             
 8   postal_code      object             
 9   browser          object             
 10  traffic_source   object             
 11  uri              object             
 12  event_type       object             
dtypes: datetime64[ns, UTC](1), float64(1), int64(2), object(9)
memory usage: 241.2+ MB


In [ ]:
events.isnull().sum()

In [ ]:
events.head()

### drop unnessecsary columns for better performance

In [8]:
columns_to_drop = ['id', 'ip_address', 'city', 'postal_code']

events = events.drop(columns=columns_to_drop)

In [ ]:
events

In [ ]:
events.isnull().sum()

### fill NaN values

In [9]:
events['user_id'] = events['user_id'].fillna(-1).astype('int64')



In [ ]:
events.isnull().sum()

In [ ]:
events.info()

In [10]:
events.head()

,user_id,sequence_number,session_id,created_at,state,browser,traffic_source,uri,event_type
0,-1,2,0eeed9b6-fc32-4c12-bb29-8e2f055c92fd,2019-07-25 12:38:00+00:00,Georgia,Chrome,Email,/product/849,product
1,-1,3,ca08193f-d7b4-4a12-9af3-d66345a387af,2023-01-21 14:08:00+00:00,Guangdong,Firefox,Email,/cart,cart
2,16837,3,356cdc19-43a3-4c4b-a743-8c22d37c4255,2020-08-27 02:55:52+00:00,Nordrhein-Westfalen,IE,Email,/cart,cart
3,62494,9,195bee73-1e1f-496e-ba55-8e7d4cc2e029,2023-03-26 10:55:54+00:00,Guangdong,Chrome,Email,/cart,cart
4,68904,4,537b4232-52c8-4bc2-82be-2712987f878f,2022-06-02 10:23:43+00:00,Guangdong,Chrome,Email,/cart,cart


### load to warehouse

In [11]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")


start_time = time.time()



print(f"Write {len(events):,} rows...")
events.to_sql(
    name='events', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f" Done! in: {round(end_time - start_time, 2)} seconds.")

try connect to trino...
Write 2,431,963 rows...


---
 Done! in: 3453.08 seconds.


In [48]:
import gc

del events
gc.collect()

print("RAM cleared!")

RAM cleared!


## 3. inventory_items

In [13]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.inventory_items"
inv_items = pd.read_sql(query, con=engine)

print('Loaded!')


In [14]:
inv_items.info()

In [22]:
inv_items.isnull().sum()


In [18]:
inv_items.head()

### convert String to Datetime (ISO8601)

In [19]:
inv_items['created_at'] = pd.to_datetime(inv_items['created_at'], format='ISO8601')
inv_items['sold_at'] = pd.to_datetime(inv_items['sold_at'], format='ISO8601')

### round to 2 decimal places

In [ ]:
inv_items['cost'] = inv_items['cost'].round(2)
inv_items['product_retail_price'] = inv_items['product_retail_price'].round(2)

### fill NaN values

In [20]:
inv_items['product_name'] = inv_items['product_name'].fillna('Unknown')
inv_items['product_brand'] = inv_items['product_brand'].fillna('Unknown')

In [24]:
inv_items.info()

### load to warehouse

In [26]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")


start_time = time.time()



print(f"Write {len(inv_items):,} rows...")
inv_items.to_sql(
    name='inventory_items', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f" Done! in: {round(end_time - start_time, 2)} seconds.")

## 4. order_items

In [27]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.order_items"
order_items = pd.read_sql(query, con=engine)

print('Loaded!')


In [28]:
order_items.info()

In [29]:
order_items.isnull().sum()

In [30]:
order_items.head()


### convert String to Datetime (ISO8601)

In [31]:
date_columns = ['created_at', 'shipped_at', 'delivered_at', 'returned_at']

for col in date_columns:
    order_items[col] = pd.to_datetime(order_items[col], format='ISO8601')

### round to 2 decimal places

In [ ]:
order_items['sale_price'] = order_items['sale_price'].round(2)

In [32]:
order_items.info()

In [34]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")


start_time = time.time()



print(f"Write {len(order_items):,} rows...")
order_items.to_sql(
    name='order_items', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f" Done! in: {round(end_time - start_time, 2)} seconds.")

## 5. orders

In [38]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.orders"
orders = pd.read_sql(query, con=engine)

print('Loaded!')


Loaded!


In [39]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125226 entries, 0 to 125225
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   order_id      125226 non-null  int64 
 1   user_id       125226 non-null  int64 
 2   status        125226 non-null  object
 3   gender        125226 non-null  object
 4   created_at    125226 non-null  object
 5   returned_at   12530 non-null   object
 6   shipped_at    81461 non-null   object
 7   delivered_at  43884 non-null   object
 8   num_of_item   125226 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 8.6+ MB


In [41]:
orders.isnull().sum()

order_id             0
user_id              0
status               0
gender               0
created_at           0
returned_at     112696
shipped_at       43765
delivered_at     81342
num_of_item          0
dtype: int64

In [42]:
orders.head()

,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,3939,3124,Cancelled,M,2023-11-06 17:23:00+00:00,None,None,None,2
1,3962,3145,Cancelled,M,2023-12-21 01:02:00+00:00,None,None,None,3
2,3965,3148,Cancelled,M,2024-01-13 14:29:00+00:00,None,None,None,1
3,3979,3160,Cancelled,M,2021-08-31 03:26:00+00:00,None,None,None,1
4,4003,3180,Cancelled,M,2021-11-06 01:58:00+00:00,None,None,None,2


### convert String to Datetime (ISO8601)

In [43]:
order_date_columns = ['created_at', 'returned_at', 'shipped_at', 'delivered_at']

for col in order_date_columns:
    orders[col] = pd.to_datetime(orders[col], format='ISO8601')

orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125226 entries, 0 to 125225
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   order_id      125226 non-null  int64              
 1   user_id       125226 non-null  int64              
 2   status        125226 non-null  object             
 3   gender        125226 non-null  object             
 4   created_at    125226 non-null  datetime64[ns, UTC]
 5   returned_at   12530 non-null   datetime64[ns, UTC]
 6   shipped_at    81461 non-null   datetime64[ns, UTC]
 7   delivered_at  43884 non-null   datetime64[ns, UTC]
 8   num_of_item   125226 non-null  int64              
dtypes: datetime64[ns, UTC](4), int64(3), object(2)
memory usage: 8.6+ MB


### load to warehouse

In [45]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")

start_time = time.time()

print(f"Write {len(orders):,} rows...")

orders.to_sql(
    name='orders', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f"Done! in: {round(end_time - start_time, 2)} seconds.")

try connect to trino...
Write 125,226 rows...


---
Done! in: 139.33 seconds.


## 6. products

In [49]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.products"
products = pd.read_sql(query, con=engine)

print('Loaded!')


In [50]:
products.info()

In [51]:
products.isnull().sum()

In [52]:
products.head()

### round to 2 decimal places

In [ ]:
products['cost'] = products['cost'].round(2)
products['retail_price'] = products['retail_price'].round(2)

### load to warehouse

In [53]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")

start_time = time.time()

print(f"Write {len(products):,} rows...")

products.to_sql(
    name='products', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f"Done! in: {round(end_time - start_time, 2)} seconds.")

## 7. users

In [54]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.users"
users = pd.read_sql(query, con=engine)

print('Loaded!')


Loaded!


In [55]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              100000 non-null  int64  
 1   first_name      100000 non-null  object 
 2   last_name       100000 non-null  object 
 3   email           100000 non-null  object 
 4   age             100000 non-null  int64  
 5   gender          100000 non-null  object 
 6   state           100000 non-null  object 
 7   street_address  100000 non-null  object 
 8   postal_code     100000 non-null  object 
 9   city            99042 non-null   object 
 10  country         100000 non-null  object 
 11  latitude        100000 non-null  float64
 12  longitude       100000 non-null  float64
 13  traffic_source  100000 non-null  object 
 14  created_at      100000 non-null  object 
dtypes: float64(2), int64(2), object(11)
memory usage: 11.4+ MB


In [56]:
users.isnull().sum()

id                  0
first_name          0
last_name           0
email               0
age                 0
gender              0
state               0
street_address      0
postal_code         0
city              958
country             0
latitude            0
longitude           0
traffic_source      0
created_at          0
dtype: int64

In [57]:
)users.head()

,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at
0,90482,Margaret,Morgan,margaretmorgan@example.com,32,F,Pernambuco,70473 Thomas Loop,56400-000,Floresta,Brasil,-8.564553,-38.305733,Search,2021-07-09 01:23:00+00:00
1,8370,Frank,Gordon,frankgordon@example.net,62,M,Pernambuco,55623 Green Groves Suite 822,56000-000,Salgueiro,Brasil,-8.132323,-39.078242,Search,2023-04-07 16:59:00+00:00
2,17853,Jonathan,Christian,jonathanchristian@example.org,39,M,Pernambuco,33357 Ariel Field,56000-000,Salgueiro,Brasil,-8.132323,-39.078242,Email,2022-12-18 14:13:00+00:00
3,19668,Leah,Collins,leahcollins@example.net,63,F,Pernambuco,8459 Kristen Row Suite 077,56000-000,Salgueiro,Brasil,-8.132323,-39.078242,Search,2020-10-09 15:47:00+00:00
4,20324,Tonya,Morgan,tonyamorgan@example.net,48,F,Pernambuco,8990 Khan Street Suite 926,56000-000,Salgueiro,Brasil,-8.132323,-39.078242,Organic,2019-06-10 04:01:00+00:00


### convert String to Datetime (ISO8601)

In [58]:
users['created_at'] = pd.to_datetime(users['created_at'], format='ISO8601')

users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   id              100000 non-null  int64              
 1   first_name      100000 non-null  object             
 2   last_name       100000 non-null  object             
 3   email           100000 non-null  object             
 4   age             100000 non-null  int64              
 5   gender          100000 non-null  object             
 6   state           100000 non-null  object             
 7   street_address  100000 non-null  object             
 8   postal_code     100000 non-null  object             
 9   city            99042 non-null   object             
 10  country         100000 non-null  object             
 11  latitude        100000 non-null  float64            
 12  longitude       100000 non-null  float64            
 13  traffic_source 

### load to warehouse

In [60]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")

start_time = time.time()

print(f"Write {len(users):,} rows...")

users.to_sql(
    name='users', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f"Done! in: {round(end_time - start_time, 2)} seconds.")

try connect to trino...
Write 100,000 rows...


---
Done! in: 175.94 seconds.


# External Datasets:

## 1. us_gas_prices

In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.us_gas_prices"
gas = pd.read_sql(query, con=engine)

print('Loaded!')


In [2]:
gas.info()

In [4]:
gas.isnull().sum()

In [5]:
gas.head()


### convert String to Datetime (ISO8601)

In [7]:
gas['date'] = pd.to_datetime(gas['date'], format='ISO8601')



### round to 2 decimal places

In [ ]:
gas['gasoline_regular'] = gas['gasoline_regular'].round(2)
gas['diesel'] = gas['diesel'].round(2)

In [ ]:
gas.info()

### load to warehouse

In [9]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")

start_time = time.time()

print(f"Write {len(gas):,} rows...")

gas.to_sql(
    name='us_gas_prices', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f"Done! in: {round(end_time - start_time, 2)} seconds.")

### 2. us_holidays

In [10]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("trino://admin@trino:8080/iceberg")

query = "SELECT * FROM raw.us_holidays"

us_holidays = pd.read_sql(query, con=engine)

print('Loaded!')

In [11]:
us_holidays.head()

In [12]:
us_holidays.info()

### convert String to Datetime (ISO8601)

In [13]:
us_holidays['date'] = pd.to_datetime(us_holidays['date'], format='ISO8601')

In [14]:
us_holidays.info()

In [18]:
import pandas as pd
from sqlalchemy import create_engine
import time

print("try connect to trino...")
engine = create_engine("trino://admin@trino:8080/iceberg")

start_time = time.time()


print(f"Write {len(us_holidays):,} rows...")

us_holidays.to_sql(
    name='us_holidays', 
    con=engine, 
    schema='warehouse',
    index=False, 
    chunksize=2000,   
    method='multi'    
)

end_time = time.time()
print("---")
print(f"Done! in: {round(end_time - start_time, 2)} seconds.")